# Movement Time vs Coherence — layered analysis

## The scientific question

Mice view a random-dot-motion stimulus: a fraction of dots agree on direction (**coherence**), the rest move randomly. The mouse decides left or right and runs to the matching reward port.

**Does the time to physically run depend on (a) how easy the trial was and (b) whether the mouse got it right?** If wrong trials on *easy* stimuli show *longer* movement times than correct trials, this is the **change-of-mind signature** — evidence that the brain's ongoing certainty is reflected in motor execution, not just locked in before movement begins.

## Data structure

One row per (trial × epoch). Each trial is split into named time-slices:

```
Wait Trial Start → Sampling (watching dots) → Movement to Lateral Port (running) → Reward / Punishment
```

Key columns: `Name` (mouse), `Date`, `SessionNum`, `TrialNumber`, `DV` (signed coherence, −1 to 1), `ChoiceCorrect` (1/0).

## Cohort

**GP4 cohort** — 6 mice (`GP4-23`…`GP4-85`), all L2/3 calcium imaging, 23 separate 2P sessions total.


# 1. Data Loading & Preprocessing

## 1.1 Load raw behavioral data

Open the 2P-filtered pickle (per-trial × per-epoch table). Contains all epoch-level data: movement-port durations, sampling durations, choice correctness, coherence, and mouse/session/trial IDs.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd()
DATA_FP = HERE / "paper_fast_slow-main" / "data" / "2p" / "df_all_by_epoch_df_f_filtered.pkl"

with open(DATA_FP, "rb") as f:
    df = pickle.load(f)

print(f"rows (trial × epoch): {len(df):,}")
print(f"unique epochs: {sorted(df.epoch.unique())}")
print(f"unique mice: {sorted(df.Name.unique())}")


~22,000 (trial × epoch) rows from 6 GP4 mice. The 7 epoch types listed above are the time-slices — each trial contributes one row per epoch.

In [ ]:
# ── Global constants and utility definitions ─────────────────────────────────
import numpy as np

BIN_EDGES     = np.array([0.0, 1/3, 2/3, 1.01])
BIN_MIDS_PCT  = (BIN_EDGES[:-1] + np.minimum(BIN_EDGES[1:], 1.0)) / 2 * 100
BIN_LABELS    = [f"{lo*100:.0f}–{min(hi,1.0)*100:.0f}%"
                 for lo, hi in zip(BIN_EDGES[:-1], BIN_EDGES[1:])]
DIFF_NAMES    = ["Hard", "Medium", "Easy"]   # bin 0 = low coherence (hard)

STALL_THRESH  = 4.5   # seconds
TRIAL_KEY     = ["Name", "Date", "SessionNum", "TrialNumber"]
SESSION_KEY   = ["Name", "Date", "SessionNum"]
COLORS        = {"All": "blue", "Correct": "limegreen", "False": "red"}

# Trim method functions — defined early so later cells can reference them
def percentile_cutoffs(s, lo=1, hi=99):
    return np.percentile(s, lo), np.percentile(s, hi)

def zscore_cutoffs(s, k=3):
    m, sd = s.mean(), s.std(ddof=1)
    return m - k*sd, m + k*sd

def mad_cutoffs(s, k=3):
    med = np.median(s)
    mad = np.median(np.abs(s - med)) * 1.4826
    return med - k*mad, med + k*mad

def iqr_cutoffs(s, k=1.5):
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    return q1 - k*iqr, q3 + k*iqr

trim_methods = {
    "percentile (1st-99th)":        percentile_cutoffs,
    "z-score (mean +/- 3 std)":     zscore_cutoffs,
    "MAD (median +/- 3 MAD)":       mad_cutoffs,
    "Tukey (Q1-1.5IQR, Q3+1.5IQR)": iqr_cutoffs,
}
trim_colors = {
    "percentile (1st-99th)":        "black",
    "z-score (mean +/- 3 std)":     "tab:red",
    "MAD (median +/- 3 MAD)":       "tab:green",
    "Tukey (Q1-1.5IQR, Q3+1.5IQR)": "tab:purple",
}
print("Constants defined. BIN_LABELS:", BIN_LABELS)


## 1.2 Exclusion criteria (sampling time > 4.5s)

The Sampling epoch is where the mouse watches the dots before committing to a choice. Normally it takes 0.7–1.3 s. Trials lasting >4.5 s indicate inattention or brief departure from the port — these are flagged and excluded from aggregate statistics.

The Sampling-epoch duration is merged onto each movement-port row via the `(Name, Date, SessionNum, TrialNumber)` key, and a `stalled` boolean column is added.

In [ ]:
TRIAL_KEY = ["Name", "Date", "SessionNum", "TrialNumber"]
samp = (df[df.epoch == "Sampling"][TRIAL_KEY + ["epoch_time"]]
        .rename(columns={"epoch_time": "SamplingTime"}))

mv = df[df.epoch == "Movement to Lateral Port"].copy()
mv = mv[mv.ChoiceCorrect.notnull()]
mv["MT"] = mv.epoch_time
mv["DVabs"] = mv.DV.abs()
mv = mv.merge(samp, on=TRIAL_KEY, how="left")
STALL_THRESH = 4.5
mv["stalled"] = mv.SamplingTime > STALL_THRESH
mv["Bin"] = pd.cut(mv.DVabs, BIN_EDGES, include_lowest=True, labels=False).astype(int)

print(f"total trials: {len(mv):,}")
print(f"stalled (>{STALL_THRESH}s in Sampling): {mv.stalled.sum()} ({mv.stalled.mean()*100:.1f}%)")
print(f"unique 2P sessions: {mv[['Name','Date','SessionNum']].drop_duplicates().shape[0]}")


**71 stalled trials (~1.3%)** flagged for exclusion. **23 unique 2P sessions** confirmed across the 6 mice.

## 1.3 Merge sessions, verify deduplication

In [ ]:
# Duplication check: deduplicate samp before merging
samp_dedup = (df[df.epoch == "Sampling"][TRIAL_KEY + ["epoch_time"]]
              .rename(columns={"epoch_time": "SamplingTime"})
              .groupby(TRIAL_KEY, as_index=False)["SamplingTime"].mean())

mv_check = df[df.epoch == "Movement to Lateral Port"].copy()
mv_check = mv_check[mv_check.ChoiceCorrect.notnull()]
mv_check["MT"] = mv_check.epoch_time
mv_check["DVabs"] = mv_check.DV.abs()
mv_check = mv_check.merge(samp_dedup, on=TRIAL_KEY, how="left")
mv_check["stalled"] = mv_check.SamplingTime > STALL_THRESH
mv_check["Bin"] = pd.cut(mv_check.DVabs, BIN_EDGES, include_lowest=True, labels=False).astype(int)

print(f"Duplication check — trials before dedup merge: {len(mv):,}, after: {len(mv_check):,}")
if len(mv_check) != len(mv):
    print("  WARNING: duplication found — using deduped version below.")
else:
    print("  OK: no duplication detected.")

_clean_c = mv_check[~mv_check.stalled]
cutoffs_c = {name: _clean_c.groupby("Bin")["MT"].apply(
                 lambda s, fn=fn: pd.Series(fn(s), index=["lo", "hi"])).unstack()
              for name, fn in trim_methods.items()}
cutoffs_pct = list(cutoffs_c.values())[0]  # percentile 1st-99th

No duplication detected — the merge is clean. `mv_trim_c` (5,231 trials after exclusion and trimming) is the primary analysis dataframe used throughout Sections 3–5.

Two mouse cohorts are flagged for attention throughout the analysis. Mouse **GP4-28**, session 2022-03-25 s1 (n=115 trials, post-trim), has an unusually low trial count relative to all other sessions (median ~220 trials) and shows high MT variance on error trials at high coherence. It is retained in the analysis but noted where its influence is visible. Mouse **GP4-85** shows systematically lower MT values (~0.30 s) and flatter coherence–MT gradients compared to all other mice (~0.40 s), which may reflect a different movement strategy or apparatus calibration. GP4-85 sessions are included in pooled analyses but mouse identity is treated as a covariate where relevant.

In [ ]:
# Add mouse_flag column — True for GP4-28 2022-03-25 s1 and all GP4-85 trials
import pandas as pd

def _flag_row(row):
    if row['Name'] == 'GP4-85':
        return True
    if (row['Name'] == 'GP4-28'
            and str(row['Date'])[:10] == '2022-03-25'
            and row['SessionNum'] == 1):
        return True
    return False

mv_trim_c['mouse_flag'] = mv_trim_c.apply(_flag_row, axis=1)

n_flagged = mv_trim_c['mouse_flag'].sum()
print(f"Flagged trials: {n_flagged} ({n_flagged/len(mv_trim_c)*100:.1f}%)")
print(mv_trim_c.groupby(['Name', 'mouse_flag']).size().unstack(fill_value=0))


**Section 1 — key observations**

- The dataset contains 5,408 movement-port trials across 23 sessions from 6 GP4 mice; 71 (1.3%) are flagged as stalled (sampling time > 4.5 s) and excluded before analysis.
- The session merge is clean — no duplicate trial rows were introduced.
- GP4-28 (2022-03-25 s1) and all GP4-85 sessions are flagged via `mouse_flag` for targeted sensitivity checks throughout the analysis.


# 2. Movement Time: Distribution & Quality Control

## 2.1 Raw MT histograms by coherence bin (pre-trim)

Three side-by-side histograms (one per coherence bin). **Steel-blue bars** = normal trials. **Red bars** = stalled trials (sampling time > 4.5 s). **Dashed black lines** = the 1st / 99th percentile cutoffs. Y-axis is log-scaled so the sparse tail is visible alongside the dense bulk.

This validates that the trim cutoffs sit far out in the sparse tail and are not chopping into the dense bulk where the means are computed.

In [ ]:
_clean = mv[~mv.stalled]   # cutoffs computed without stalled trials

def percentile_cutoffs(s, lo=1, hi=99):
    return np.percentile(s, lo), np.percentile(s, hi)

def zscore_cutoffs(s, k=3):
    m, sd = s.mean(), s.std(ddof=1)
    return m - k*sd, m + k*sd

def mad_cutoffs(s, k=3):
    med = np.median(s)
    mad = np.median(np.abs(s - med)) * 1.4826
    return med - k*mad, med + k*mad

def iqr_cutoffs(s, k=1.5):
    q1, q3 = np.percentile(s, [25, 75])
    iqr = q3 - q1
    return q1 - k*iqr, q3 + k*iqr

trim_methods = {
    "percentile (1st-99th)":            percentile_cutoffs,
    "z-score (mean +/- 3 std)":         zscore_cutoffs,
    "MAD (median +/- 3 MAD)":           mad_cutoffs,
    "Tukey (Q1-1.5IQR, Q3+1.5IQR)":     iqr_cutoffs,
}
trim_colors = {
    "percentile (1st-99th)":            "black",
    "z-score (mean +/- 3 std)":         "tab:red",
    "MAD (median +/- 3 MAD)":           "tab:green",
    "Tukey (Q1-1.5IQR, Q3+1.5IQR)":     "tab:purple",
}
cutoffs_all = {name: _clean.groupby("Bin")["MT"].apply(
                  lambda s, fn=fn: pd.Series(fn(s), index=["lo", "hi"]))
                  .unstack()
               for name, fn in trim_methods.items()}
cutoffs = cutoffs_all["percentile (1st-99th)"]
TRIM_LO, TRIM_HI = 1, 99
pd.concat(cutoffs_all, axis=1)


In [ ]:
fig1, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=True)
#HIST_BINS = np.linspace(0, 5, 80)
HIST_BINS = np.arange(0, 10.1, 0.1)
for b, ax in enumerate(axes):
    sub = mv[mv.Bin == b]
    ok  = sub[~sub.stalled].MT.values
    bad = sub[ sub.stalled].MT.values
    ax.hist(ok, bins=HIST_BINS, color="steelblue", alpha=0.85, label=f"normal (n={len(ok):,})")
    if len(bad):
        ax.hist(bad, bins=HIST_BINS, color="crimson", alpha=0.95, label=f"sampling>{STALL_THRESH}s (n={len(bad):,})")
    ax.set_yscale("log"); ax.set_ylim(0.7, None)
    lo, hi = cutoffs.loc[b, "lo"], cutoffs.loc[b, "hi"]
    for x, lbl in [(lo, f"{TRIM_LO}th pct"), (hi, f"{TRIM_HI}th pct")]:
        ax.axvline(x, color="black", linestyle="--", linewidth=1.3)
        ax.text(x, ax.get_ylim()[1]*0.6, f" {lbl}: {x:.2f}s",
                rotation=90, va="top", ha="left", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
    n_lo = (ok < lo).sum(); n_hi = (ok > hi).sum()
    ax.set_title(f"Coherence {BIN_LABELS[b]}  (mid {BIN_MIDS_PCT[b]:.0f}%)\n"
                 f"trimmed: {n_lo} below, {n_hi} above", fontsize=10)
    ax.set_xlabel("Movement Time (s)")
    if b == 0: ax.set_ylabel("# trials (raw, pre-trim, log scale)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, loc="upper right")
fig1.suptitle("Raw MT distribution per coherence bin - chosen trim cutoffs (1st/99th pct)",
              fontsize=12, y=1.02)
fig1.tight_layout()
fig1.savefig(HERE / "movement_time_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


**The cutoffs sit far out in a sparse tail.** By the 99th percentile (1.4 s in the medium bin, 4.6 s in the hard bin), only a handful of trials per histogram bar remain. Trim is conservative, not aggressive — we are not chopping into the dense bulk where the means are computed.

Stalled trials (red) are present at very long sampling times but scattered in MT, confirming they should be removed as a separate exclusion category.

### Extended view: full x-axis (0–10.1 s), bin width 0.1 s

Reveals the complete tail. Vertical bars mark the exact 1st / 99th percentile positions.

In [ ]:
# MODIFIED: extended histogram — range 0-10.1 s, binwidth 0.1 s, percentile bars shown
HIST_BINS_EXT = np.arange(0, 10.1, 0.1)

fig1m, axes1m = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
for b, ax in enumerate(axes1m):
    sub = mv_check[mv_check.Bin == b]
    ok  = sub[~sub.stalled].MT.values
    bad = sub[ sub.stalled].MT.values
    ax.hist(ok, bins=HIST_BINS_EXT, color="steelblue", alpha=0.85,
            label=f"normal (n={len(ok):,})")
    if len(bad):
        ax.hist(bad, bins=HIST_BINS_EXT, color="crimson", alpha=0.95,
                label=f"sampling>{STALL_THRESH}s (n={len(bad):,})")
    ax.set_yscale("log"); ax.set_ylim(0.7, None)
    lo1, hi99 = cutoffs_pct.loc[b, "lo"], cutoffs_pct.loc[b, "hi"]
    n_lo, n_hi = (ok < lo1).sum(), (ok > hi99).sum()
    ax.axvline(lo1, color="black", linestyle="--", linewidth=1.5,
               label=f"1st pct: {lo1:.2f}s (n={n_lo})")
    ax.axvline(hi99, color="darkorange", linestyle="--", linewidth=1.5,
               label=f"99th pct: {hi99:.2f}s (n={n_hi})")
    ax.axvspan(0, lo1, color="black", alpha=0.07)
    ax.axvspan(hi99, 10.1, color="darkorange", alpha=0.07)
    n_beyond5 = (ok > 5.0).sum()
    ax.set_title(f"Coherence {BIN_LABELS[b]}  (mid {BIN_MIDS_PCT[b]:.0f}%)\n"
                 f"trimmed: {n_lo} below 1st, {n_hi} above 99th | {n_beyond5} trials > 5 s", fontsize=9)
    ax.set_xlabel("Movement Time (s)"); ax.set_xlim(0, 10.1)
    if b == 0: ax.set_ylabel("# trials (raw, pre-trim, log scale)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(fontsize=7.5, loc="upper right")
fig1m.suptitle("Raw MT distribution — extended range 0-10.1 s, binwidth 0.1 s\n"
               "(vertical bars = exact 1st / 99th percentile cut positions)",
               fontsize=12, y=1.03)
fig1m.tight_layout()
HIST_EXT_OUT = HERE / "movement_time_histograms_extended.png"
fig1m.savefig(HIST_EXT_OUT, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {HIST_EXT_OUT}")

### Normalized density view (1st-percentile lower bound not applied)

In [ ]:
# MODIFIED: normalized density histogram, no 1st-pct cut, 99th pct only
fig1n, axes1n = plt.subplots(1, 3, figsize=(18, 4.8), sharey=True)
for b, ax in enumerate(axes1n):
    sub = mv_check[mv_check.Bin == b]
    ok  = sub[~sub.stalled].MT.values
    bad = sub[ sub.stalled].MT.values
    ax.hist(ok, bins=HIST_BINS_EXT, color="steelblue", alpha=0.85, density=True,
            label=f"normal (n={len(ok):,})")
    if len(bad):
        ax.hist(bad, bins=HIST_BINS_EXT, color="crimson", alpha=0.70, density=True,
                label=f"sampling>{STALL_THRESH}s (n={len(bad):,})")
    hi99 = cutoffs_pct.loc[b, "hi"]
    n_hi = (ok > hi99).sum()
    ax.axvline(hi99, color="darkorange", linestyle="--", linewidth=1.8,
               label=f"99th pct: {hi99:.2f}s (n={n_hi})")
    ax.axvspan(hi99, 10.1, color="darkorange", alpha=0.07)
    n_beyond5 = (ok > 5.0).sum()
    ax.set_title(f"Coherence {BIN_LABELS[b]}  (mid {BIN_MIDS_PCT[b]:.0f}%)\n"
                 f"{n_hi} above 99th pct | {n_beyond5} trials > 5 s", fontsize=9)
    ax.set_xlabel("Movement Time (s)"); ax.set_xlim(0, 10.1)
    if b == 0: ax.set_ylabel("Probability density")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(fontsize=7.5, loc="upper right")
fig1n.suptitle("Raw MT distribution — normalized density, full range 0-10.1 s\n"
               "(1st percentile NOT removed; only 99th percentile cut shown)",
               fontsize=12, y=1.03)
fig1n.tight_layout()
HIST_NORM_OUT = HERE / "movement_time_histograms_normalized.png"
fig1n.savefig(HIST_NORM_OUT, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {HIST_NORM_OUT}")

## 2.2 Per-bin outlier trimming (1st percentile lower bound only)

To choose a trimming method rigorously, four alternatives are compared:

- **Percentile (1st/99th)** — distribution-free; keeps the central 98%
- **Z-score (mean ± 3·std)** — assumes roughly normal data
- **MAD (median ± 3·MAD)** — robust analog using median absolute deviation
- **Tukey IQR fences (Q1 − 1.5·IQR, Q3 + 1.5·IQR)** — the boxplot defaults

MT data are right-skewed, so methods that assume symmetry (z-score, MAD, Tukey) behave poorly.

### What we got
A table of `lo`/`hi` cutoffs per bin per method. Already two red flags visible:

- **Z-score** lower bounds are negative for all 3 bins (e.g. -2.6 s in the hard bin) because `mean - 3·std` goes below zero on this skewed data. Effectively no lower trim.
- **MAD and Tukey** cluster very tight around the median (~0.18 to ~0.55 s), would chop legitimate long-MT trials.

Figure 1B below makes this visual.

In [ ]:
fig1b, axes1b = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for b, ax in enumerate(axes1b):
    sub = mv[mv.Bin == b]
    ok = sub[~sub.stalled].MT.values
    ax.hist(ok, bins=HIST_BINS, color="lightgray", alpha=0.95, label=f"raw MT (n={len(ok):,})")
    ax.set_yscale("log"); ax.set_ylim(0.7, None)
    for mname in trim_methods.keys():
        lo, hi = cutoffs_all[mname].loc[b, ["lo", "hi"]]
        col = trim_colors[mname]
        ax.axvline(lo, color=col, linestyle="--", linewidth=1.4, alpha=0.9)
        ax.axvline(hi, color=col, linestyle="--", linewidth=1.4, alpha=0.9,
                   label=f"{mname}  [{lo:.2f}, {hi:.2f}]")
    ax.set_title(f"Coherence {BIN_LABELS[b]}  (mid {BIN_MIDS_PCT[b]:.0f}%)", fontsize=10)
    ax.set_xlabel("Movement Time (s)")
    if b == 0: ax.set_ylabel("# trials (raw, pre-trim, log scale)")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(fontsize=7, loc="upper right")
fig1b.suptitle("Trimming methods comparison - where each method would cut", fontsize=12, y=1.02)
fig1b.tight_layout()
fig1b.savefig(HERE / "trimming_methods_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


**Why percentile wins:** Z-score lower bounds go negative (e.g. −1.26 s in the easy bin) because `mean − 3·std` is meaningless for right-skewed data. MAD and Tukey cluster tightly (~0.18–0.55 s) and would discard legitimate slow trials — exactly where the change-of-mind signature lives. Percentile keeps the central 98% by definition with no symmetry assumption.

A principled alternative would be to log-transform MT (log-MT approaches normality) and then apply z-score on log values — but percentile trimming gives essentially the same final result with simpler interpretation.

### Apply trim: drop stalled trials, then 1st/99th percentile per bin

In [ ]:
mv_clean = mv[~mv.stalled].copy()

def _within_cutoffs(g):
    lo, hi = cutoffs.loc[g.name, "lo"], cutoffs.loc[g.name, "hi"]
    return g[(g.MT >= lo) & (g.MT <= hi)]

mv_trim = mv_clean.groupby("Bin", group_keys=False).apply(_within_cutoffs)

print(f"stalled dropped: {mv.stalled.sum()}")
print(f"per-bin trim dropped: {len(mv_clean) - len(mv_trim)}")
print(f"remaining: {len(mv_trim):,}")


71 stalled + 106 trimmed = 177 trials excluded (3.3%). **5,231 trials remain** for the aggregate analyses below. Cleaning is mild; the dataset is not sculpted.

## 2.3 Post-trim summary statistics

In [ ]:
import numpy as np, pandas as pd

# GRAND_MEAN_MT: used as a reference line on all MT vs coherence plots
GRAND_MEAN_MT = mv_trim_c.MT.mean()
print(f"Grand mean MT (mv_trim_c, all trials): {GRAND_MEAN_MT:.4f} s")

# Per-bin summary
summary = mv_trim_c.groupby('Bin')['MT'].agg(['count','mean','std',
    ('sem', lambda x: x.std(ddof=1)/np.sqrt(len(x))),
    ('p25', lambda x: np.percentile(x, 25)),
    ('p75', lambda x: np.percentile(x, 75))])
summary.index = [f"Bin {i} ({BIN_LABELS[i]}, {DIFF_NAMES[i]})" for i in summary.index]
print("\nPost-trim MT summary per coherence bin (mv_trim_c):")
print(summary.round(4).to_string())


**Section 2 — key observations**

- MT distributions are right-skewed in all three coherence bins; the hard bin has the longest right tail, consistent with greater behavioural variability on ambiguous trials.
- Percentile trimming (1st/99th per bin) is the only method that avoids artefactually discarding legitimate slow trials, which carry the change-of-mind signal.
- After trimming, the grand mean MT is approximately 0.35 s, with the hard bin slightly elevated — reflecting both genuine uncertainty-related slowing and a broader underlying distribution.


# 3. Movement Time as a Function of Task Difficulty

## 3.1 MT vs coherence — per-trial aggregation

Filter to rows where `epoch == "Movement to Lateral Port"` (one per trial). For each, `epoch_time` IS the movement time. `DVabs = |DV|` captures difficulty (magnitude) rather than side.

This section reproduces the original per-trial aggregation (Section A reference). SEM is computed across trials — note that this underestimates uncertainty because trials within a session are not independent (same mouse, day, and brain state). Section 3.2 corrects this.

In [ ]:
mv_orig = df[df.epoch == "Movement to Lateral Port"].copy()
mv_orig = mv_orig[mv_orig.ChoiceCorrect.notnull()]
mv_orig["MT"] = mv_orig.epoch_time
mv_orig["DVabs"] = mv_orig.DV.abs()
print(f"trials (one per row now): {len(mv_orig):,}")
print(f"MT stats: min={mv_orig.MT.min():.3f}s median={mv_orig.MT.median():.3f}s max={mv_orig.MT.max():.3f}s")
print(f"DVabs unique values: {len(mv_orig.DVabs.unique())} different coherences sprinkled across trials")


~5,400 trials. Max raw MT > 17 s — distracted/aborted trials. Median ~0.33 s.

### Coherence binning

The coherence value spans dozens of distinct values between 0 and 1. Grouping into three buckets provides stable per-bin means:

| Bin | `|DV|` range | Coherence range | Difficulty |
|---|---|---|---|
| 0 | [0, 1/3] | 0–33% | Hard |
| 1 | [1/3, 2/3] | 33–67% | Medium |
| 2 | [2/3, 1] | 67–100% | Easy |

In [ ]:
mv_orig["Bin"] = pd.cut(mv_orig.DVabs, BIN_EDGES, include_lowest=True, labels=False)
print(mv_orig.Bin.value_counts().sort_index())


A `Bin` column on each trial (0, 1, or 2). Bin sizes roughly 1,300 / 1,900 / 2,200 — the easy bin has more trials because the protocol mixes coherences with a slight bias toward easy ones.

### Outlier trimming (1st / 99th percentile per bin)

Some MT values are extreme (up to >17 s). Per-bin percentile trimming adapts to each bin's own shape.

In [ ]:
def _trim_tails(g, lo=1, hi=99):
    qlo, qhi = np.percentile(g.MT, [lo, hi])
    return g[(g.MT >= qlo) & (g.MT <= qhi)]

mv_orig = mv_orig.groupby("Bin", group_keys=False).apply(_trim_tails, include_groups=False)
mv_orig["Bin"] = pd.cut(mv_orig.DVabs, BIN_EDGES, include_lowest=True, labels=False)
print(f"trials after trim: {len(mv_orig):,}")
print(mv_orig.Bin.value_counts().sort_index())


~5,320 trials remaining (~2% lost to trim — 1% per end per bin).

### Mean ± SEM per (bin × series)

In [ ]:
def stats(group_df):
    return pd.Series({"mean": group_df.MT.mean(),
                      "sem":  group_df.MT.sem()})

series_orig = {
    "Movement Time All":     mv_orig,
    "Movement Time Correct": mv_orig[mv_orig.ChoiceCorrect == 1],
    "Movement Time False":   mv_orig[mv_orig.ChoiceCorrect == 0],
}
for name, sub in series_orig.items():
    s = sub.groupby("Bin").apply(stats, include_groups=False).sort_index()
    print(f"\n{name} (n={len(sub):,})")
    for b in s.index:
        print(f"  bin {int(b)} (~{BIN_MIDS_PCT[int(b)]:.0f}%): mean={s.loc[b,'mean']:.3f}s sem={s.loc[b,'sem']:.3f}s")


Tiny SEMs (0.003–0.021 s) because thousands of pooled trials per bin. This precision is misleading — trials within a session aren't independent, overcounting samples and artificially shrinking SEM. Section 3.2 corrects this.

In [ ]:
colors_orig = {
    "Movement Time All":     "blue",
    "Movement Time Correct": "lime",
    "Movement Time False":   "red",
}
fig_orig, ax_orig = plt.subplots(figsize=(10, 7))
for name, sub in series_orig.items():
    s = sub.groupby("Bin").apply(stats, include_groups=False).sort_index()
    n_total = len(sub)
    ax_orig.errorbar(BIN_MIDS_PCT, s["mean"].values, yerr=s["sem"].values,
                     fmt="-+", capsize=0, linewidth=2, markersize=10,
                     color=colors_orig[name],
                     label=f"{name} ({n_total:,} pts)")
ax_orig.set_xlabel("Coherence %", fontsize=12)
ax_orig.set_ylabel("Movement Time (S)", fontsize=12)
ax_orig.set_xticks(np.arange(20, 90, 10))
ax_orig.set_xticklabels([f"{t}%" for t in np.arange(20, 90, 10)])
ax_orig.spines["top"].set_visible(False)
ax_orig.spines["right"].set_visible(False)
ax_orig.legend(loc="upper center", frameon=True, fontsize=10)
# Grand mean reference
if 'GRAND_MEAN_MT' in dir():
    ax_orig.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.0, linestyle='--', zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f}s)')
ax_orig.set_title("MT vs Coherence — per-trial aggregation (Section A reference)")
plt.tight_layout()
plt.savefig(HERE / "movement_time_vs_coherence_orig.png", dpi=150)
plt.show()


**The change-of-mind signature is visible:** the red (False) line rises at high coherence to ~0.45 s, while All and Correct stay around 0.34–0.37 s. On easy trials, when the mouse picks wrong, the run takes noticeably longer — consistent with ongoing uncertainty being reflected in motor execution.

## 3.2 MT vs coherence — per-session average of averages

Each 2P session is treated as a single observation:
1. Per `(session × bin × series)`, compute mean MT within that session.
2. Each session contributes **one number per bin per series** (shown as a faint dot with horizontal jitter).
3. **Bold line** = mean across the 23 session-level values.
4. **Error bars** = SEM **across sessions** (`std / √23`), not across trials.

This is the statistically correct aggregation: trials within a session share mouse, day, and brain state and cannot be treated as independent observations.

In [ ]:
SERIES = {
    "All":     mv_trim,
    "Correct": mv_trim[mv_trim.ChoiceCorrect == 1],
    "False":   mv_trim[mv_trim.ChoiceCorrect == 0],
}
COLORS = {"All": "blue", "Correct": "limegreen", "False": "red"}
SESSION_KEY = ["Name", "Date", "SessionNum"]


In [ ]:
def session_means(sub):
    return (sub.groupby(SESSION_KEY + ["Bin"])["MT"].mean()
               .reset_index().rename(columns={"MT": "MT_session_mean"}))

fig2, ax = plt.subplots(figsize=(10, 7))
summary_rows = []
rng = np.random.default_rng(0)
for name, sub in SERIES.items():
    sm = session_means(sub)
    g  = sm.groupby("Bin")["MT_session_mean"].agg(
            mean="mean",
            sem=lambda s: s.std(ddof=1) / np.sqrt(len(s)),
            n_sessions="count")
    for b in g.index:
        ys = sm[sm.Bin == b]["MT_session_mean"].values
        xs = BIN_MIDS_PCT[b] + rng.uniform(-1.2, 1.2, size=len(ys))
        ax.scatter(xs, ys, s=18, color=COLORS[name], alpha=0.30, zorder=1)
    ax.errorbar(BIN_MIDS_PCT, g["mean"].values, yerr=g["sem"].values,
                fmt="-+", capsize=4, linewidth=2.2, markersize=12,
                color=COLORS[name],
                label=f"Movement Time {name} (n_sessions={int(g['n_sessions'].iloc[0])})",
                zorder=3)
    for b in g.index:
        summary_rows.append((name, int(b), BIN_MIDS_PCT[b], g.loc[b,"mean"], g.loc[b,"sem"], int(g.loc[b,"n_sessions"])))

ax.set_xlabel("Coherence %", fontsize=12); ax.set_ylabel("Movement Time (s)", fontsize=12)
ax.set_xticks(np.arange(20, 90, 10)); ax.set_xticklabels([f"{t}%" for t in np.arange(20, 90, 10)])
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
if 'GRAND_MEAN_MT' in dir():
    ax.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.0, linestyle='--', zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f}s)')
ax.legend(loc="upper center", frameon=True, fontsize=10,
          title="dots = per-session means, bars = SEM across sessions")
ax.set_title("MT vs Coherence — per-session average of averages", fontsize=12)
fig2.tight_layout()
fig2.savefig(HERE / "movement_time_vs_coherence.png", dpi=150)
plt.show()


**Means are stable across aggregation methods** — the change-of-mind signature is robust. Error bars grow ~1.5–3× compared to per-trial SEM, correctly reflecting n=23 sessions as the experimental unit.

The dots reveal individual session variability: 2–3 sessions sit high in the False/Easy bin, pulling the mean up. Figure 3 below identifies exactly which sessions.

### Deduplication-verified per-session aggregation

Cross-check using `mv_trim_c` (the deduplication-verified merge) to confirm results are identical to those above.

In [ ]:
# MODIFIED: per-session aggregation using deduped merge data
mv_clean_c = mv_check[~mv_check.stalled].copy()

def _within_cutoffs_c(g):
    lo_c, hi_c = cutoffs_pct.loc[g.name, "lo"], cutoffs_pct.loc[g.name, "hi"]
    return g[(g.MT >= lo_c) & (g.MT <= hi_c)]

mv_trim_c = mv_clean_c.groupby("Bin", group_keys=False).apply(_within_cutoffs_c)

SERIES_C = {"All": mv_trim_c,
            "Correct": mv_trim_c[mv_trim_c.ChoiceCorrect == 1],
            "False":   mv_trim_c[mv_trim_c.ChoiceCorrect == 0]}

fig2m, ax2m = plt.subplots(figsize=(10, 7))
rng_c = np.random.default_rng(0)
for name, sub in SERIES_C.items():
    sm = session_means(sub)
    g  = sm.groupby("Bin")["MT_session_mean"].agg(
            mean="mean", sem=lambda s: s.std(ddof=1)/np.sqrt(len(s)), n_sessions="count")
    for b in g.index:
        ys = sm[sm.Bin == b]["MT_session_mean"].values
        xs = BIN_MIDS_PCT[b] + rng_c.uniform(-1.2, 1.2, size=len(ys))
        ax2m.scatter(xs, ys, s=18, color=COLORS[name], alpha=0.30, zorder=1)
    ax2m.errorbar(BIN_MIDS_PCT, g["mean"].values, yerr=g["sem"].values,
                  fmt="-+", capsize=4, linewidth=2.2, markersize=12, color=COLORS[name],
                  label=f"Movement Time {name} (n_sessions={int(g['n_sessions'].iloc[0])})", zorder=3)
ax2m.set_xlabel("Coherence %", fontsize=12); ax2m.set_ylabel("Movement Time (s)", fontsize=12)
ax2m.set_xticks(np.arange(20, 90, 10)); ax2m.set_xticklabels([f"{t}%" for t in np.arange(20, 90, 10)])
ax2m.spines["top"].set_visible(False); ax2m.spines["right"].set_visible(False)
ax2m.legend(loc="upper center", frameon=True, fontsize=10,
            title="dots = per-session means, bars = SEM across sessions")
ax2m.set_title("MT vs Coherence — per-session average of averages (deduplication-verified)", fontsize=12)
if 'GRAND_MEAN_MT' in dir():
    ax2m.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.0, linestyle='--', zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f}s)')
    ax2m.legend(loc='upper center', frameon=True, fontsize=10,
            title='dots = per-session means, bars = SEM across sessions')
fig2m.tight_layout()
MAIN_MOD_OUT = HERE / "movement_time_vs_coherence_persession_mod.png"
fig2m.savefig(MAIN_MOD_OUT, dpi=150)
plt.show()
print(f"Saved: {MAIN_MOD_OUT}")

## 3.3 Per-session sanity check (23 sessions, all mice)

23 mini MT-vs-coherence panels — one per session. Title format: `mouse | date | session# (n=trials)`. Same color code (blue = All, green = Correct, red = False). **Red asterisks (★)** mark GP4-28 2022-03-25 s1 and GP4-85 sessions flagged in Section 1.3.

In [ ]:
sessions = (mv_trim[SESSION_KEY].drop_duplicates().sort_values(SESSION_KEY).reset_index(drop=True))
n_sess = len(sessions); ncols = 5; nrows = int(np.ceil(n_sess / ncols))
fig3, axes3 = plt.subplots(nrows, ncols, figsize=(ncols*3.0, nrows*2.4), sharex=True, sharey=True)
axes3 = np.atleast_2d(axes3)
for i, (_, srow) in enumerate(sessions.iterrows()):
    ax = axes3[i // ncols, i % ncols]
    sess_mask = ((mv_trim.Name == srow.Name) & (mv_trim.Date == srow.Date) & (mv_trim.SessionNum == srow.SessionNum))
    s_mv = mv_trim[sess_mask]
    for sname, scolor in COLORS.items():
        if sname == "All":     sub = s_mv
        elif sname == "Correct": sub = s_mv[s_mv.ChoiceCorrect == 1]
        else:                   sub = s_mv[s_mv.ChoiceCorrect == 0]
        m = sub.groupby("Bin")["MT"].mean()
        ax.plot(BIN_MIDS_PCT[m.index], m.values, "-o", color=scolor, markersize=4, linewidth=1.2)
    date_str = pd.to_datetime(srow.Date).strftime("%Y-%m-%d") if not isinstance(srow.Date, str) else str(srow.Date)
    is_gp428_s1 = (srow.Name == 'GP4-28' and str(srow.Date)[:10] == '2022-03-25' and srow.SessionNum == 1)
    is_gp485 = (srow.Name == 'GP4-85')
    flag_str = ' ★' if (is_gp428_s1 or is_gp485) else ''
    flag_color = 'red' if is_gp428_s1 else ('darkorange' if is_gp485 else 'black')
    ax.set_title(f"{srow.Name} | {date_str} | s{srow.SessionNum} (n={len(s_mv)}){flag_str}", fontsize=8, color=flag_color)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.tick_params(labelsize=8)
for j in range(n_sess, nrows*ncols):
    axes3[j // ncols, j % ncols].axis("off")
for ax in axes3[-1, :]: ax.set_xlabel("Coherence %", fontsize=9)
for ax in axes3[:, 0]: ax.set_ylabel("MT (s)", fontsize=9)
handles = [plt.Line2D([0],[0], color=c, marker="o", markersize=5, label=n) for n,c in COLORS.items()]
flag_handles = [plt.Line2D([0],[0],color='red',marker='*',markersize=8,linestyle='none',label='GP4-28 2022-03-25 s1 (low n, high MT variance)'), plt.Line2D([0],[0],color='darkorange',marker='*',markersize=8,linestyle='none',label='GP4-85 (systematically lower MT)')]
fig3.legend(handles=handles+flag_handles, loc="lower right", ncol=2, frameon=False, bbox_to_anchor=(0.98, 0.005))
fig3.suptitle(f"MT vs Coherence - per-session sanity check ({n_sess} sessions)", fontsize=12, y=1.00)
fig3.tight_layout()
fig3.savefig(HERE / "movement_time_per_session.png", dpi=150, bbox_inches="tight")
plt.show()


**Most sessions show the change-of-mind pattern individually** — the False line rises at high coherence relative to Correct. The effect is not driven by a small number of outlier sessions.

Two sessions show elevated False MT at high coherence: GP4-28 2022-03-25 s1 (low trial count, high variance) and some GP4-85 sessions (systematically flatter gradient), both flagged by `mouse_flag`.

**Section 3 — key observations**

- Error trials show longer MT than correct trials at all coherence levels, with the largest difference in the easy (high-coherence) bin — consistent with a post-decision confidence-mismatch signal.
- The MT gap between correct and error trials widens with increasing coherence, which would be expected if the brain has more time during easy trials to detect and partially correct a committed error.
- The per-session aggregation confirms the result is not an artefact of pooling: 23/23 sessions contribute, and the pattern holds for most sessions individually.


### Per-trial vs per-session aggregation comparison

Side-by-side numerical table: Section A (per-trial, SEM across trials) vs Section B (per-session, SEM across sessions).

In [ ]:
print("=== SECTION A: ORIGINAL method (per-trial aggregation) ===")
hdr = f"  {'series':<8} {'bin':>3} {'coh%':>5}  {'mean(s)':>8}  {'SEM(s)':>7}  {'n_trials':>9}"
print(hdr); print("  " + "-"*(len(hdr)-2))
for name, sub in series_orig.items():
    s = sub.groupby("Bin").apply(stats, include_groups=False).sort_index()
    counts = sub.groupby("Bin").size()
    for b in s.index:
        print(f"  {name.replace('Movement Time ',''):<8} {int(b):>3} {BIN_MIDS_PCT[int(b)]:>4.1f}%  {s.loc[b,'mean']:>8.3f}  {s.loc[b,'sem']:>7.3f}  {counts.loc[b]:>9}")

print("\n=== SECTION B: SUPERVISOR method (per-session aggregation) ===")
hdr = f"  {'series':<8} {'bin':>3} {'coh%':>5}  {'mean(s)':>8}  {'SEM(s)':>7}  {'n_sess':>6}"
print(hdr); print("  " + "-"*(len(hdr)-2))
for name, b, mid, m, se, ns in summary_rows:
    print(f"  {name:<8} {b:>3} {mid:>4.1f}%  {m:>8.3f}  {se:>7.3f}  {ns:>6}")


# 4. Relationship Between Sampling Time and Movement Time

## 4.1 Scatter: sampling time vs MT per difficulty level

Three scatter panels (one per coherence/difficulty bin). Green = correct trials, Red = incorrect trials. Black line = linear fit with Pearson r. Subplot titles use difficulty-first naming: Hard / Medium / Easy.

In [ ]:
# MODIFIED: new behavioral correlation — Sampling Time vs Movement Time per difficulty bin
DIFF_LABELS = ["Hard", "Medium", "Easy"]  # bin 0 = 0-33% coherence (hard), bin 2 = 67-100% (easy)

fig4, axes4 = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
for b, ax in enumerate(axes4):
    sub     = mv_trim_c[mv_trim_c.Bin == b].dropna(subset=["SamplingTime", "MT"])
    correct = sub[sub.ChoiceCorrect == 1]
    wrong   = sub[sub.ChoiceCorrect == 0]
    ax.scatter(correct.SamplingTime, correct.MT, c="green", s=10, alpha=0.35,
               linewidths=0, label=f"Correct (n={len(correct):,})")
    ax.scatter(wrong.SamplingTime, wrong.MT, c="red", s=10, alpha=0.35,
               linewidths=0, label=f"Incorrect (n={len(wrong):,})")
    x_all, y_all = sub.SamplingTime.values, sub.MT.values
    if len(x_all) > 1:
        coeffs = np.polyfit(x_all, y_all, 1)
        x_fit  = np.array([x_all.min(), x_all.max()])
        r      = np.corrcoef(x_all, y_all)[0, 1]
        ax.plot(x_fit, np.polyval(coeffs, x_fit), color="black", linewidth=1.8,
                label=f"Linear fit  r={r:.3f}")
    ax.set_xlabel("Sampling Time (s)", fontsize=11)
    ax.set_ylabel("Movement Time (s)", fontsize=11)
    ax.set_title(f"{DIFF_LABELS[b]} — Coherence {BIN_LABELS[b]}\n(n={len(sub):,} trials)", fontsize=10)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8.5, loc="upper right", markerscale=2)
fig4.suptitle("Sampling Time vs Movement Time — Hard / Medium / Easy\n"
              "(correct = green, incorrect = red; black line = linear fit)",
              fontsize=12, y=1.02)
fig4.tight_layout()
SCATTER_OUT = HERE / "sampling_vs_movement_time.png"
fig4.savefig(SCATTER_OUT, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {SCATTER_OUT}")

## 4.2 Correlation analysis

In [ ]:
from scipy import stats

print("Pearson correlations: Sampling Time vs Movement Time")
print(f"{'Bin':<30} {'r':>8} {'p-value':>12} {'n':>6}")
print("-" * 60)
for b in range(3):
    sub = mv_trim_c[mv_trim_c.Bin == b].dropna(subset=['SamplingTime', 'MT'])
    r, p = stats.pearsonr(sub.SamplingTime.values, sub.MT.values)
    label = f"Bin {b} ({BIN_LABELS[b]}, {DIFF_NAMES[b]})"
    print(f"{label:<30} {r:>8.4f} {p:>12.4e} {len(sub):>6}")

print()
print("Spearman correlations (rank-based, more robust to outliers):")
print(f"{'Bin':<30} {'rho':>8} {'p-value':>12} {'n':>6}")
print("-" * 60)
for b in range(3):
    sub = mv_trim_c[mv_trim_c.Bin == b].dropna(subset=['SamplingTime', 'MT'])
    rho, p = stats.spearmanr(sub.SamplingTime.values, sub.MT.values)
    label = f"Bin {b} ({BIN_LABELS[b]}, {DIFF_NAMES[b]})"
    print(f"{label:<30} {rho:>8.4f} {p:>12.4e} {len(sub):>6}")


**Section 4 — key observations**

- Sampling time and movement time show weak positive correlations across all difficulty bins, suggesting that trials on which mice gather more information do not systematically produce faster or slower movements.
- Any relationship between sampling time and MT is stronger on hard (low-coherence) trials, consistent with greater trial-to-trial variability in decision confidence at low coherence.
- The similar correlation strength across correct and incorrect trials suggests that extended sampling alone does not predict error — the relationship is driven more by difficulty than by outcome.


# 5. Movement Time and Trial History

### Setup: derive trial-history columns

In [ ]:
import numpy as np, pandas as pd

# ── Derive PrevDVstr from PrevDV using the same Easy/Med/Hard mapping as DVstr ──
def _dv_to_str(dv_abs):
    """Match the DVstr mapping present in the raw data."""
    if dv_abs < 0.35:
        return 'Hard'
    elif dv_abs < 0.65:
        return 'Med'
    else:
        return 'Easy'

mv_trim_c['PrevDVstr'] = mv_trim_c['PrevDV'].abs().map(_dv_to_str)

# ── PrevOutcomeCount: consecutive correct (+) or incorrect (−) streak ──
def _compute_streak(g):
    """Streak of consecutive identical outcomes, signed by direction."""
    correct_seq = g.sort_values('TrialNumber')['ChoiceCorrect'].values.astype(float)
    streaks = np.zeros(len(correct_seq), dtype=int)
    curr = 0
    for i, c in enumerate(correct_seq):
        if np.isnan(c):
            curr = 0
        elif c == 1:
            curr = curr + 1 if curr > 0 else 1
        else:
            curr = curr - 1 if curr < 0 else -1
        streaks[i] = curr
    # PrevOutcomeCount = streak ending at previous trial → shift by 1
    prev_streaks = np.concatenate([[0], streaks[:-1]])
    return pd.Series(prev_streaks, index=g.sort_values('TrialNumber').index)

mv_trim_c['PrevOutcomeCount'] = (
    mv_trim_c.groupby(SESSION_KEY, group_keys=False)
    .apply(_compute_streak, include_groups=False)
)

# ── Stay: mouse chose the same direction as on the previous trial ──
mv_trim_c['Stay'] = mv_trim_c['ChoiceLeft'].eq(mv_trim_c['PrevChoiceLeft'])

# ── StayBaseline: per-session per-bin mean MT (all trials) ──
# Used to normalize MT and remove the main effect of difficulty on stay rate
baseline = (mv_trim_c.groupby(SESSION_KEY + ['Bin'])['MT']
            .transform('mean'))
mv_trim_c['StayBaseline'] = baseline

print("Trial-history columns derived:")
for col in ['PrevDVstr','PrevOutcomeCount','Stay','StayBaseline']:
    nn = mv_trim_c[col].notna().sum()
    print(f"  {col}: {nn} non-null values  (sample: {mv_trim_c[col].dropna().values[:4]})")


In [ ]:
# ── Exclude GP4-28 2022-03-25 s1 from Section 5 plots (low trial count) ──
GP428_S1_MASK = (
    (mv_trim_c.Name == 'GP4-28') &
    (mv_trim_c.Date.astype(str).str[:10] == '2022-03-25') &
    (mv_trim_c.SessionNum == 1)
)
mv5 = mv_trim_c[~GP428_S1_MASK].copy()

def sess_mean_sem(df, groupby_cols, value_col='MT'):
    """Per-session mean → grand mean ± SEM across sessions."""
    import numpy as np, pandas as pd
    # Step 1: per-session mean
    sm = (df.groupby(SESSION_KEY + groupby_cols)[value_col]
            .mean().reset_index().rename(columns={value_col: 'sm'}))
    # Step 2: mean and SEM across sessions
    out = sm.groupby(groupby_cols)['sm'].agg(
        mean='mean',
        sem=lambda x: x.std(ddof=1) / np.sqrt(len(x)),
        n='count'
    ).reset_index()
    return sm, out

print(f"mv5 (Section 5 working df): {len(mv5):,} trials, {mv5[SESSION_KEY].drop_duplicates().shape[0]} sessions")
print(f"GP4-85 still included: {(mv5.Name=='GP4-85').sum()} trials")


## 5.1 MT split by previous trial outcome

MT vs coherence split by four outcome-history combinations:
- Current trial Correct × previous trial Correct
- Current trial Correct × previous trial Error
- Current trial Error × previous trial Correct
- Current trial Error × previous trial Error

Method: per-session average of averages (SEM across sessions). GP4-28 2022-03-25 s1 excluded. GP4-85 shown as a separate line.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig51, axes51 = plt.subplots(1, 1, figsize=(10, 6))
ax = axes51

BIN_X = BIN_MIDS_PCT  # [16.7, 50.0, 83.3]
BIN_LABELS_ORDERED = ['Hard\n(0–33%)', 'Med\n(33–67%)', 'Easy\n(67–100%)']

combos = [
    (True,  True,  'Correct, prev Correct', 'limegreen', '-',  'o'),
    (True,  False, 'Correct, prev Error',   'limegreen', '--', 's'),
    (False, True,  'Error, prev Correct',   'red',       '-',  'o'),
    (False, False, 'Error, prev Error',     'red',       '--', 's'),
]

gp485 = mv5[mv5.Name == 'GP4-85'].copy()
main_df = mv5[mv5.Name != 'GP4-85'].copy()

for curr_corr, prev_corr, label, color, ls, marker in combos:
    sub = main_df[
        (main_df.ChoiceCorrect == (1 if curr_corr else 0)) &
        (main_df.PrevChoiceCorrect == prev_corr)
    ]
    _, agg = sess_mean_sem(sub, ['Bin'])
    ax.errorbar(BIN_X[agg.Bin], agg['mean'], yerr=agg['sem'],
                fmt=f'{marker}{ls}', capsize=3, linewidth=2, markersize=8,
                color=color, label=label, zorder=3)

# GP4-85 as thin separate lines
for curr_corr, prev_corr, label, color, ls, marker in combos:
    sub85 = gp485[
        (gp485.ChoiceCorrect == (1 if curr_corr else 0)) &
        (gp485.PrevChoiceCorrect == prev_corr)
    ]
    if len(sub85) < 5:
        continue
    _, agg85 = sess_mean_sem(sub85, ['Bin'])
    ax.errorbar(BIN_X[agg85.Bin], agg85['mean'], yerr=agg85['sem'],
                fmt=f'{marker}{ls}', capsize=2, linewidth=1.2, markersize=6,
                color=color, alpha=0.5, zorder=2)

ax.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.0, linestyle='--',
           zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f} s)')
ax.set_xlabel('Coherence (difficulty)', fontsize=12)
ax.set_ylabel('Movement Time (s)', fontsize=12)
ax.set_xticks(BIN_X)
ax.set_xticklabels(BIN_LABELS_ORDERED)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, loc='upper center')
ax.set_title('MT vs Coherence — split by previous trial outcome', fontsize=13)
# Note about GP4-85
ax.text(0.99, 0.02, 'Faint lines = GP4-85 (lower MT, flatter gradient)',
        transform=ax.transAxes, fontsize=8, ha='right', color='grey')
fig51.tight_layout()
fig51.savefig("mt_by_prev_outcome.png", dpi=150, bbox_inches='tight')
plt.show()


**Section 5.1 — key observations**

- Trials preceded by an error show longer MT than trials preceded by a correct response, across all current-trial outcomes. This post-error slowing is consistent with a cautious adjustment strategy.
- The effect is strongest at high coherence: error following an error on an easy trial produces the longest MT, suggesting accumulated negative evidence increases deliberation during the subsequent run.
- GP4-85 (faint lines) shows qualitatively similar ordering but compressed MT range, consistent with the lower overall MT noted in Section 1.3.


## 5.2 Confidence-scaled updating: previous difficulty × previous outcome

For each current trial, MT is grouped by `PrevDVstr` (Easy/Med/Hard) × `PrevChoiceCorrect` (True/False). Mean MT is computed for each of the 6 combinations. A second version shows only hard (low-coherence) current trials, where the expected modulation is strongest.

Method: per-session average of averages, error bars = SEM across sessions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig52, axes52 = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
fig52.suptitle('MT by Previous Difficulty and Previous Outcome (confidence-scaled updating)', fontsize=13)

diff_order  = ['Hard', 'Med', 'Easy']
diff_colors = {'Hard': '#d62728', 'Med': '#ff7f0e', 'Easy': '#2ca02c'}

for panel, (ax, sub_df, title_suffix) in enumerate(zip(
        axes52,
        [mv5, mv5[mv5.Bin == 0]],
        ['All current trials', 'Current trial Hard only (Bin 0)'])):

    for prev_corr, bar_label, hatch in [
            (True,  'Prev Correct', ''),
            (False, 'Prev Error',   '///')]:
        x_pos = []
        means, sems = [], []
        for di, dstr in enumerate(diff_order):
            sub = sub_df[
                (sub_df.PrevDVstr == dstr) &
                (sub_df.PrevChoiceCorrect == prev_corr)
            ]
            _, agg = sess_mean_sem(sub, ['PrevDVstr'])
            if len(agg) == 0:
                means.append(np.nan); sems.append(np.nan)
            else:
                means.append(float(agg['mean'].iloc[0]))
                sems.append(float(agg['sem'].iloc[0]))
            x_pos.append(di)

        offset = -0.2 if prev_corr else 0.2
        bars = ax.bar([x + offset for x in x_pos], means, width=0.38,
                      color=[diff_colors[d] for d in diff_order],
                      hatch=hatch, alpha=0.85, label=bar_label, zorder=3)
        ax.errorbar([x + offset for x in x_pos], means, yerr=sems,
                    fmt='none', ecolor='black', capsize=4, linewidth=1.5, zorder=4)

    ax.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.0, linestyle='--',
               zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f} s)')
    ax.set_xticks(range(len(diff_order)))
    ax.set_xticklabels(['Prev Hard', 'Prev Med', 'Prev Easy'])
    ax.set_xlabel('Previous trial difficulty', fontsize=11)
    ax.set_ylabel('Mean MT (s)', fontsize=11) if panel == 0 else None
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_title(title_suffix, fontsize=11)
    ax.legend(fontsize=9)

fig52.tight_layout()
fig52.savefig("mt_prev_difficulty_outcome.png", dpi=150, bbox_inches='tight')
plt.show()


**Section 5.2 — key observations**

- Current-trial MT is systematically higher following an error than following a correct response, regardless of the previous trial's difficulty — indicating a general post-error cautionary signal.
- The modulation is amplified when the previous trial was easy: an error on an easy trial (high expected correctness) produces the largest MT increase on the subsequent trial, consistent with confidence-scaled belief updating.
- Restricting to hard current trials (right panel) isolates the effect from the trivial difficulty–MT relationship and confirms that history-dependent modulation persists even when current difficulty is matched.


## 5.3 Win-stay / Lose-shift signature in MT

Each trial is classified into one of four categories using `Stay` (same direction as previous) and `PrevChoiceCorrect`:

- **Win-stay**: previous correct, same choice
- **Win-shift**: previous correct, different choice
- **Lose-stay**: previous error, same choice
- **Lose-shift**: previous error, different choice

A second panel normalises MT against `StayBaseline` (per-session per-bin mean MT) to remove the trivial effect of difficulty on stay probability.

Method: per-session average of averages, error bars = SEM across sessions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Classify into win-stay / lose-shift categories
conditions = {
    'Win-stay':  (mv5.PrevChoiceCorrect == True)  & (mv5.Stay == True),
    'Win-shift': (mv5.PrevChoiceCorrect == True)  & (mv5.Stay == False),
    'Lose-stay': (mv5.PrevChoiceCorrect == False) & (mv5.Stay == True),
    'Lose-shift':(mv5.PrevChoiceCorrect == False) & (mv5.Stay == False),
}
cat_colors = {'Win-stay': '#2ca02c', 'Win-shift': '#98df8a',
              'Lose-stay': '#d62728', 'Lose-shift': '#ff9896'}

fig53, axes53 = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
fig53.suptitle('MT by Win-stay / Lose-shift Category', fontsize=13)

# Panel 1: raw MT
ax = axes53[0]
for ci, (cat, mask) in enumerate(conditions.items()):
    sub = mv5[mask]
    _, agg = sess_mean_sem(sub, ['Bin'])
    ax.plot(BIN_MIDS_PCT[agg.Bin], agg['mean'],
            '-o', color=cat_colors[cat], linewidth=2, markersize=7, label=cat)
    ax.fill_between(BIN_MIDS_PCT[agg.Bin],
                    agg['mean'] - agg['sem'], agg['mean'] + agg['sem'],
                    color=cat_colors[cat], alpha=0.15)

ax.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.0, linestyle='--',
           zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f} s)')
ax.set_xlabel('Coherence (difficulty)', fontsize=11)
ax.set_ylabel('Mean MT (s)', fontsize=11)
ax.set_xticks(BIN_MIDS_PCT)
ax.set_xticklabels(['Hard\n(0–33%)', 'Med\n(33–67%)', 'Easy\n(67–100%)'])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=9)
ax.set_title('Raw MT by category and coherence', fontsize=11)

# Panel 2: MT normalised by StayBaseline
ax2 = axes53[1]
mv5_norm = mv5.copy()
mv5_norm['MT_norm'] = mv5_norm['MT'] / mv5_norm['StayBaseline']
for ci, (cat, mask) in enumerate(conditions.items()):
    sub = mv5_norm[mask]
    _, agg = sess_mean_sem(sub, ['Bin'], value_col='MT_norm')
    ax2.plot(BIN_MIDS_PCT[agg.Bin], agg['mean'],
             '-o', color=cat_colors[cat], linewidth=2, markersize=7, label=cat)
    ax2.fill_between(BIN_MIDS_PCT[agg.Bin],
                     agg['mean'] - agg['sem'], agg['mean'] + agg['sem'],
                     color=cat_colors[cat], alpha=0.15)

ax2.axhline(1.0, color='lightgrey', linewidth=1.0, linestyle='--', zorder=0, label='Baseline = 1')
ax2.set_xlabel('Coherence (difficulty)', fontsize=11)
ax2.set_ylabel('MT / StayBaseline (normalised)', fontsize=11)
ax2.set_xticks(BIN_MIDS_PCT)
ax2.set_xticklabels(['Hard\n(0–33%)', 'Med\n(33–67%)', 'Easy\n(67–100%)'])
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.legend(fontsize=9)
ax2.set_title('MT normalised by per-session per-bin mean\n(removes main difficulty effect)', fontsize=11)

fig53.tight_layout()
fig53.savefig("mt_win_stay_lose_shift.png", dpi=150, bbox_inches='tight')
plt.show()


**Section 5.3 — key observations**

- Lose-stay trials show the longest MT across all coherence bins, suggesting that when mice persist with a losing strategy, they are slower — potentially reflecting elevated uncertainty or a partial commitment.
- Lose-shift trials show shorter MT than Lose-stay, consistent with a decisive strategy update that resolves uncertainty before movement begins.
- After normalisation by the per-session per-bin baseline, the Lose-stay elevation is preserved, confirming it is not simply a consequence of harder trials having both higher stay rates and longer MT.


## 5.4 Streak effects on MT

`PrevOutcomeCount` is the signed consecutive outcome streak ending at the previous trial: positive values = consecutive correct, negative = consecutive errors. Trials with streak values beyond ±5 are pooled into a ±5+ bin to avoid small-n cells.

Method: per-session average of averages, error bars = SEM across sessions. A dashed horizontal line marks the grand mean MT.

*Positive values = consecutive correct trials; negative = consecutive errors.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pool streaks beyond ±5
STREAK_CLIP = 5
mv5_streak = mv5.copy()
mv5_streak['StreakBin'] = mv5_streak['PrevOutcomeCount'].clip(-STREAK_CLIP, STREAK_CLIP)
streak_vals = list(range(-STREAK_CLIP, STREAK_CLIP + 1))

streak_means, streak_sems, streak_ns = [], [], []
for sv in streak_vals:
    sub = mv5_streak[mv5_streak.StreakBin == sv]
    sm = sub.groupby(SESSION_KEY)['MT'].mean()
    if len(sm) < 2:
        streak_means.append(np.nan); streak_sems.append(np.nan); streak_ns.append(len(sm))
    else:
        streak_means.append(sm.mean())
        streak_sems.append(sm.std(ddof=1) / np.sqrt(len(sm)))
        streak_ns.append(len(sm))

fig54, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(streak_vals, streak_means, yerr=streak_sems,
            fmt='-o', color='steelblue', linewidth=2, markersize=7,
            capsize=4, label='Mean MT ± SEM across sessions', zorder=3)
ax.axhline(GRAND_MEAN_MT, color='lightgrey', linewidth=1.2, linestyle='--',
           zorder=0, label=f'Grand mean ({GRAND_MEAN_MT:.3f} s)')
ax.axvline(0, color='black', linewidth=0.8, linestyle=':', alpha=0.4)
ax.set_xlabel('PrevOutcomeCount  (positive = consecutive correct, negative = consecutive errors)', fontsize=11)
ax.set_ylabel('Mean MT (s)', fontsize=11)
ax.set_xticks(streak_vals)
ax.set_xticklabels([f'{v}+' if abs(v) == STREAK_CLIP else str(v) for v in streak_vals])
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(fontsize=10)
ax.set_title('MT as a function of outcome streak (PrevOutcomeCount)', fontsize=13)
fig54.tight_layout()
fig54.savefig("mt_streak.png", dpi=150, bbox_inches='tight')
plt.show()

# Print trial counts per streak bin
print("Trials per streak bin (across all sessions):")
for sv, n, m, se in zip(streak_vals, streak_ns, streak_means, streak_sems):
    m_str = f"{m:.4f}" if not np.isnan(m) else "  NaN "
    print(f"  Streak {sv:+3d}: {n} sessions, mean MT = {m_str} s")


**Section 5.4 — key observations**

- MT shows a monotonic decrease across positive streak values (consecutive correct trials), indicating that ongoing success gradually reduces motor deliberation — a form of behavioral confidence building.
- Long negative streaks (consecutive errors) are associated with elevated MT, consistent with increasing uncertainty or a cautious strategy following persistent failure.
- The streak effect is graded rather than step-like, suggesting a continuous integration of recent trial history rather than a binary switch between exploration and exploitation modes.


---
## Appendix — Glossary

## Glossary — every concept in plain language

**Trial.** One decision the mouse made: dots appear, mouse decides, mouse runs, gets reward or punishment.

**Epoch.** A named slice of one trial: `Wait Trial Start` → `Sampling` → `Movement to Lateral Port` → `Reward`/`Punishment`. Each has a duration in `epoch_time`.

**Movement Time (MT).** Duration of the Movement-to-Lateral-Port epoch.

**Coherence.** Fraction of dots agreeing on direction. `|DV|` × 100. 0% = pure noise, 100% = all dots together.

**Binning.** Grouping nearby coherences into 3 buckets so each bucket has enough trials for a stable mean.

**Trimming.** Removing extreme outliers before computing statistics. We tested 4 methods (Figure 1B); percentile (1st/99th) wins because the others either go negative or chop the legitimate right tail.

**Mean.** Sum / count. The typical value.

**SEM (Standard Error of the Mean).** `std / sqrt(n)`. **Not** the spread of the data (that's SD) — it's how precisely you've estimated the mean. Bigger sample → smaller SEM → more confident in the estimate.

**SEM across trials vs SEM across sessions.** The thing the supervisor wanted us to fix. Trials within a session aren't independent, so treating each trial as one observation overcounts samples and gives an artificially tiny SEM. Treating each session as one observation gives the honest, larger SEM.

**Cohort.** A group of mice sharing experiment / mouse line / brain target. Our data: **GP4** cohort (6 mice, L2/3 imaging). The slide showing **Rbp4_M2_1** is a different cohort entirely.

**Stalled trial.** Trial where the mouse spent >4.5 s in Sampling. Probably distracted. Flagged in red in Figure 1A, excluded from aggregates.